# Sistema de Scoring Crediticio con Optimización de Rentabilidad Ajustada por Riesgo

## Contexto del Problema

El riesgo crediticio representa uno de los principales desafíos para las entidades financieras y plataformas de préstamos. Cada vez que una institución otorga un crédito, asume la posibilidad de que el cliente no cumpla con sus obligaciones de pago, generando pérdidas financieras y afectando la rentabilidad del portafolio.

Este proyecto utiliza datos históricos de LendingClub, una plataforma estadounidense de *peer-to-peer lending (P2P lending)* que conectaba inversionistas con personas que solicitaban préstamos personales. Antes de aprobar un préstamo, LendingClub evaluaba el perfil financiero y crediticio de cada solicitante con el objetivo de estimar el riesgo asociado a la operación.

El dataset contiene información sobre millones de préstamos otorgados entre 2007 y 2018, incluyendo variables relacionadas con:

- Perfil financiero del solicitante
- Historial crediticio
- Condiciones del préstamo
- Estado final del crédito

La variable objetivo principal es `loan_status`, que permite identificar si un préstamo fue pagado correctamente o terminó en incumplimiento (*default*).

---

# Objetivo del Proyecto

El objetivo general de este proyecto es desarrollar un sistema de riesgo crediticio basado en Machine Learning capaz de estimar la probabilidad de incumplimiento de un préstamo utilizando información disponible al momento de la solicitud.

A partir de estas predicciones, el proyecto busca simular estrategias de aprobación y analizar cómo distintas políticas de riesgo afectan la rentabilidad y pérdida esperada de un portafolio de créditos.

---

# Objetivos Técnicos

Los objetivos técnicos del proyecto incluyen:

- Realizar un análisis exploratorio de datos (EDA) para comprender la estructura y calidad del dataset.
- Identificar y tratar valores faltantes, variables irrelevantes y posibles problemas de *data leakage*.
- Construir pipelines de preprocesamiento para variables numéricas y categóricas.
- Entrenar y comparar modelos de clasificación para predicción de default.
- Evaluar el desempeño utilizando métricas apropiadas para problemas de riesgo crediticio, como ROC-AUC, Recall y F1-score.
- Interpretar el comportamiento del modelo mediante técnicas de importancia de variables e interpretabilidad.

---

# Objetivos Aplicados al Negocio

Desde una perspectiva financiera, el proyecto busca transformar las probabilidades generadas por el modelo en herramientas de apoyo para la toma de decisiones.

Esto incluye:

- Estimar la probabilidad de default de cada solicitante.
- Construir un sistema de *credit scoring* basado en riesgo.
- Calcular la pérdida esperada (*Expected Loss*) de los préstamos.
- Simular políticas de aprobación utilizando distintos umbrales de riesgo.
- Optimizar decisiones de otorgamiento de crédito para maximizar rentabilidad y minimizar pérdidas.

De esta manera, el proyecto no se limita únicamente a un problema de clasificación, sino que incorpora conceptos reales de modelado de riesgo financiero y optimización de decisiones.

## 1.0 Librerías

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)


pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\accepted_2007_to_2018Q4.csv", low_memory= False)

In [ ]:
dict_df = pd.read_excel(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\LCDataDictionary.xlsx")

c:\Users\camil\anaconda3\envs\mlp311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
browse_feat = dict_df["LoanStatNew"].dropna().values

In [ ]:
browse_feat = [x.replace("\xa0", "") for x in browse_feat]

correcciones = {
    "is_inc_v": "verification_status",
    "verified_status_joint": "verification_status_joint"
}

browse_feat = [correcciones.get(col, col) for col in browse_feat]

In [ ]:
valid_features = list(set(browse_feat).intersection(set(df.columns)))

In [ ]:
len(valid_features)

76

In [ ]:
df_browse = df[valid_features].copy()
df_browse["loan_status"] = df["loan_status"]

## 1.1 Entendimiento de los datos

In [ ]:
df_browse.head()

,int_rate,max_bal_bc,last_pymnt_amnt,total_rec_late_fee,il_util,initial_list_status,tot_coll_amt,policy_code,application_type,total_pymnt_inv,sub_grade,fico_range_high,open_acc,out_prncp_inv,home_ownership,issue_d,term,open_il_12m,url,collection_recovery_fee,pub_rec,revol_bal,emp_title,total_rec_int,dti,last_fico_range_high,loan_status,installment,delinq_2yrs,total_rec_prncp,recoveries,last_fico_range_low,annual_inc,title,fico_range_low,next_pymnt_d,emp_length,id,open_rv_12m,purpose,dti_joint,open_il_24m,desc,addr_state,zip_code,member_id,total_acc,open_acc_6m,funded_amnt,out_prncp,verification_status,mths_since_last_record,loan_amnt,acc_now_delinq,revol_util,annual_inc_joint,inq_fi,mths_since_last_delinq,inq_last_12m,pymnt_plan,open_rv_24m,total_pymnt,inq_last_6mths,last_pymnt_d,total_cu_tl,collections_12_mths_ex_med,grade,all_util,funded_amnt_inv,total_bal_il,mths_since_rcnt_il,earliest_cr_line,verification_status_joint,last_credit_pull_d,tot_cur_bal,mths_since_last_major_derog
0,13.99,722.0,122.67,0.0,36.0,w,722.0,1.0,Individual,4421.72,C4,679.0,7.0,0.00,MORTGAGE,Dec-2015,36 months,0.0,https://lendingclub.com/browse/loanDetail.acti...,0.0,0.0,2765.0,leadman,821.72,5.91,564.0,Fully Paid,123.03,0.0,3600.00,0.0,560.0,55000.0,Debt consolidation,675.0,NaN,10+ years,68407277,3.0,debt_consolidation,NaN,1.0,NaN,PA,190xx,NaN,13.0,2.0,3600.0,0.00,Not Verified,NaN,3600.0,0.0,29.7,NaN,3.0,30.0,4.0,n,3.0,4421.723917,1.0,Jan-2019,1.0,0.0,C,34.0,3600.0,4981.0,21.0,Aug-2003,NaN,Mar-2019,144904.0,30.0
1,11.99,6472.0,926.35,0.0,73.0,w,0.0,1.0,Individual,25679.66,C1,719.0,22.0,0.00,MORTGAGE,Dec-2015,36 months,0.0,https://lendingclub.com/browse/loanDetail.acti...,0.0,0.0,21470.0,Engineer,979.66,16.06,699.0,Fully Paid,820.28,1.0,24700.00,0.0,695.0,65000.0,Business,715.0,NaN,10+ years,68355089,2.0,small_business,NaN,1.0,NaN,SD,577xx,NaN,38.0,1.0,24700.0,0.00,Not Verified,NaN,24700.0,0.0,19.2,NaN,0.0,6.0,6.0,n,3.0,25679.660000,4.0,Jun-2016,0.0,0.0,C,29.0,24700.0,18005.0,19.0,Dec-1999,NaN,Mar-2019,204396.0,NaN
2,10.78,2081.0,15813.30,0.0,73.0,w,0.0,1.0,Joint App,22705.92,B4,699.0,6.0,0.00,MORTGAGE,Dec-2015,60 months,0.0,https://lendingclub.com/browse/loanDetail.acti...,0.0,0.0,7869.0,truck driver,2705.92,10.78,704.0,Fully Paid,432.66,0.0,20000.00,0.0,700.0,63000.0,NaN,695.0,NaN,10+ years,68341763,0.0,home_improvement,13.85,4.0,NaN,IL,605xx,NaN,18.0,0.0,20000.0,0.00,Not Verified,NaN,20000.0,0.0,56.2,71000.0,2.0,NaN,1.0,n,2.0,22705.924294,0.0,Jun-2017,5.0,0.0,B,65.0,20000.0,10827.0,19.0,Aug-2000,Not Verified,Mar-2019,189699.0,NaN
3,14.85,6987.0,829.90,0.0,70.0,w,0.0,1.0,Individual,31464.01,C5,789.0,13.0,15897.65,MORTGAGE,Dec-2015,60 months,0.0,https://lendingclub.com/browse/loanDetail.acti...,0.0,0.0,7802.0,Information Systems Officer,12361.66,17.06,679.0,Current,829.90,0.0,19102.35,0.0,675.0,110000.0,Debt consolidation,785.0,Apr-2019,10+ years,66310712,1.0,debt_consolidation,NaN,1.0,NaN,NJ,076xx,NaN,17.0,1.0,35000.0,15897.65,Source Verified,NaN,35000.0,0.0,11.6,NaN,0.0,NaN,0.0,n,1.0,31464.010000,0.0,Feb-2019,1.0,0.0,C,45.0,35000.0,12609.0,23.0,Sep-2008,NaN,Mar-2019,301500.0,NaN
4,22.45,9702.0,10128.96,0.0,84.0,w,0.0,1.0,Individual,11740.50,F1,699.0,12.0,0.00,MORTGAGE,Dec-2015,60 months,0.0,https://lendingclub.com/browse/loanDetail.acti...,0.0,0.0,21929.0,Contract Specialist,1340.50,25.37,704.0,Fully Paid,289.91,1.0,10400.00,0.0,700.0,104433.0,Major purchase,695.0,NaN,3 years,68476807,4.0,major_purchase,NaN,3.0,NaN,PA,174xx,NaN,35.0,1.0,10400.0,0.00,Source Verified,NaN,10400.0,0.0,64.5,NaN,2.0,12.0,3.0,n,7.0,11740.500000,3.0,Jul-2016,1.0,0.0,F,78.0,10400.0,73839.0,14.0,Jun-1998,NaN,Mar-2018,331730.0,NaN


In [ ]:
print(f'The dataset has {df_browse.shape[0]} rows and {df_browse.shape[1]} columns.')

The dataset has 2260701 rows and 76 columns.


## 1.2 Selección y Comprensión de Variables

El dataset original de LendingClub contiene más de 150 variables. Sin embargo, muchas de ellas corresponden a información generada después de que el préstamo fue aprobado o durante su ciclo de vida. Estas variables no estarían disponibles en el momento real de evaluar a un solicitante y, por lo tanto, introducirían problemas de *data leakage* si fueran utilizadas para entrenar el modelo.

Para identificar las variables realmente disponibles para inversionistas y analistas en el momento de la solicitud del préstamo, se utilizó el diccionario oficial de datos de LendingClub (`LCDataDictionary.xlsx`), específicamente la hoja denominada *Browse Notes*, publicada por Wendy Kan.

A partir de esta referencia, se realizó una intersección entre:

- Las variables presentes en el dataset original
- Las variables documentadas como visibles para inversionistas en el diccionario de datos

Este proceso permitió reducir el conjunto inicial de variables y conservar únicamente aquellas relevantes para un escenario realista de predicción de riesgo crediticio.

Posteriormente, durante las etapas de análisis exploratorio y modelado, algunas variables adicionales serán eliminadas debido a:

- Altos porcentajes de valores faltantes
- Baja utilidad predictiva
- Cardinalidad excesiva
- Posibles problemas de fuga de información (*data leakage*)

No obstante, ciertas variables serán conservadas temporalmente durante el EDA con el fin de entender mejor el comportamiento financiero de los préstamos, incluso si posteriormente no son utilizadas para entrenar el modelo final.


## 1.2.1 Diccionario de Variables — Lending Club

A continuación se presenta la descripción técnica y funcional de las variables utilizadas en el proyecto de riesgo crediticio con Lending Club. Las variables se agrupan según su naturaleza y utilidad dentro del análisis exploratorio, modelado predictivo y evaluación financiera.

---

### 1. Variables de Identificación y Metadatos

| Variable | Descripción |
|---|---|
| `id` | Identificador único del préstamo. | 
| `member_id` | Identificador único del cliente en Lending Club. |
| `url` | URL asociada al préstamo dentro de Lending Club. | 

---

### 2. Información del Préstamo y Condiciones Financieras

| Variable | Descripción |
|---|---|
| `loan_amnt` | Monto total solicitado por el prestatario. |
| `funded_amnt` | Monto financiado por Lending Club. |
| `funded_amnt_inv` | Monto financiado por inversionistas. |
| `term` | Duración del préstamo (36 o 60 meses). |
| `int_rate` | Tasa de interés anual aplicada al préstamo. |
| `installment` | Cuota mensual que debe pagar el prestatario. |
| `grade` | Calificación crediticia general asignada por Lending Club. |
| `sub_grade` | Subcategoría detallada de riesgo dentro del grade. |
| `purpose` | Motivo declarado del préstamo. |
| `title` | Título descriptivo asignado por el prestatario. |
| `issue_d` | Fecha de emisión del préstamo. |
| `application_type` | Tipo de aplicación: individual o conjunta (*joint application*). |
| `initial_list_status` | Estado inicial del préstamo en el marketplace (Whole/Fractional). |

---

### 3. Información Laboral y Socioeconómica

| Variable | Descripción |
|---|---|
| `emp_title` | Cargo laboral reportado por el prestatario. |
| `emp_length` | Antigüedad laboral en años. |
| `home_ownership` | Tipo de tenencia de vivienda (RENT, OWN, MORTGAGE). |
| `annual_inc` | Ingreso anual reportado por el prestatario. |
| `verification_status` | Estado de verificación de ingresos. |
| `annual_inc_joint` | Ingreso anual combinado para aplicaciones conjuntas. |
| `verification_status_joint` | Estado de verificación de ingresos en aplicaciones conjuntas. |
| `zip_code` | Código postal parcial del prestatario. |
| `addr_state` | Estado de residencia del prestatario. |

---

### 4. Variables de Riesgo Crediticio e Historial Financiero

| Variable | Descripción |
|---|---|
| `dti` | Relación deuda-ingreso (*Debt-to-Income Ratio*). |
| `dti_joint` | Relación deuda-ingreso para aplicaciones conjuntas. |
| `fico_range_low` | Límite inferior del rango FICO del prestatario. |
| `fico_range_high` | Límite superior del rango FICO del prestatario. |
| `last_fico_range_low` | Último rango FICO inferior registrado. |
| `last_fico_range_high` | Último rango FICO superior registrado. |
| `delinq_2yrs` | Número de moras en los últimos 2 años. |
| `acc_now_delinq` | Número de cuentas actualmente en mora. |
| `pub_rec` | Número de registros públicos negativos (ej. bancarrota). |
| `open_acc` | Número de cuentas de crédito abiertas. |
| `total_acc` | Número total de cuentas de crédito en historial. |
| `revol_bal` | Balance total de crédito revolvente utilizado. |
| `revol_util` | Utilización del crédito revolvente (%). |
| `inq_last_6mths` | Número de consultas de crédito en los últimos 6 meses. |
| `inq_last_12m` | Número de consultas de crédito en los últimos 12 meses. |
| `earliest_cr_line` | Fecha de apertura de la línea de crédito más antigua. |
| `mths_since_last_delinq` | Meses desde la última mora registrada. |
| `mths_since_last_record` | Meses desde el último registro público negativo. |
| `mths_since_last_major_derog` | Meses desde el último evento derogatorio severo. |

---

### 5. Variables Derivadas de Crédito e Información Bancaria

| Variable | Descripción |
|---|---|
| `tot_cur_bal` | Balance total actual de todas las cuentas. |
| `tot_coll_amt` | Total de cuentas enviadas a cobranza. |
| `total_rev_hi_lim` | Límite total de crédito revolvente disponible. |
| `total_bal_il` | Balance total de líneas de crédito a plazos (*installment loans*). |
| `all_util` | Utilización total del crédito disponible. |
| `il_util` | Utilización de líneas de crédito a plazos. |
| `max_bal_bc` | Balance máximo en tarjetas bancarias. |
| `total_cu_tl` | Número total de líneas de crédito financiero abiertas. |

---

### 6. Variables Relacionadas con Apertura y Actividad de Crédito

| Variable | Descripción |
|---|---|
| `open_acc_6m` | Número de cuentas abiertas en los últimos 6 meses. |
| `open_il_12m` | Número de líneas de crédito a plazos abiertas en 12 meses. |
| `open_il_24m` | Número de líneas de crédito a plazos abiertas en 24 meses. |
| `open_rv_12m` | Número de líneas revolventes abiertas en 12 meses. |
| `open_rv_24m` | Número de líneas revolventes abiertas en 24 meses. |
| `open_il_6m` | Número de líneas de crédito a plazos abiertas en 6 meses. |
| `mths_since_rcnt_il` | Meses desde la apertura más reciente de una línea a plazos. |
| `inq_fi` | Número de consultas financieras personales recientes. |

---

### 7. Estado del Préstamo y Variables Post-Emisión (Potential Data Leakage)

Estas variables contienen información generada después de la aprobación del préstamo y, aunque son útiles para análisis financieros y EDA, no deben utilizarse para entrenar modelos predictivos de default.

| Variable | Descripción |
|---|---|
| `loan_status` | Estado final o actual del préstamo. |
| `pymnt_plan` | Indica si existe un plan de pagos especial. |
| `out_prncp` | Principal pendiente por pagar. |
| `out_prncp_inv` | Principal pendiente correspondiente a inversionistas. |
| `total_pymnt` | Total pagado por el prestatario. |
| `total_pymnt_inv` | Total pagado a inversionistas. |
| `total_rec_prncp` | Capital recuperado. |
| `total_rec_int` | Intereses recuperados. |
| `total_rec_late_fee` | Cargos por mora recuperados. |
| `recoveries` | Monto recuperado después de default. |
| `collection_recovery_fee` | Comisiones asociadas a recuperación de deuda. |
| `last_pymnt_d` | Fecha del último pago realizado. |
| `last_pymnt_amnt` | Monto del último pago realizado. |
| `next_pymnt_d` | Próxima fecha estimada de pago. |
| `last_credit_pull_d` | Última fecha de consulta crediticia realizada por Lending Club. |

---

### 8. Variables Especiales para Aplicaciones Conjuntas (Joint Applications)

Estas variables solo aplican a préstamos realizados por más de un prestatario y presentan altos porcentajes de valores faltantes debido a que la mayoría de préstamos son individuales.

| Variable | Descripción |
|---|---|
| `annual_inc_joint` | Ingreso anual combinado de solicitantes conjuntos. |
| `dti_joint` | Relación deuda-ingreso conjunta. |
| `verification_status_joint` | Estado de verificación de ingresos conjuntos. |

---

### Consideraciones Técnicas

- Algunas variables con altos porcentajes de valores faltantes fueron conservadas inicialmente para análisis exploratorio.
- Variables como `grade` y `sub_grade` representan sistemas internos de scoring desarrollados por Lending Club y serán comparadas posteriormente con los modelos de Machine Learning desarrollados en este proyecto.

## 1.3 Información general

In [ ]:
df_browse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 76 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   int_rate                     float64
 1   max_bal_bc                   float64
 2   last_pymnt_amnt              float64
 3   total_rec_late_fee           float64
 4   il_util                      float64
 5   initial_list_status          object 
 6   tot_coll_amt                 float64
 7   policy_code                  float64
 8   application_type             object 
 9   total_pymnt_inv              float64
 10  sub_grade                    object 
 11  fico_range_high              float64
 12  open_acc                     float64
 13  out_prncp_inv                float64
 14  home_ownership               object 
 15  issue_d                      object 
 16  term                         object 
 17  open_il_12m                  float64
 18  url                          object 
 19  

 - Con un primer acercamiento encontramos qué podemos eliminar variables sin valor predictivo, ni analítico como lo son:
     - `id`
     - `member_id`
     - `url`

- También se identificaron variables con posible *data leakage*.
  - Estas variables contienen información generada después de la emisión del préstamo.
  - Por tanto, no pueden utilizarse en un modelo de originación crediticia (*application credit scoring*).

- Entre las variables con *data leakage* se encuentran:
  - `recoveries`
  - `collection_recovery_fee`
  - `total_rec_prncp`
  - `total_rec_int`
  - `last_pymnt_amnt`
  - `last_pymnt_d`
  - `next_pymnt_d`

- Estas variables podrían inflar artificialmente el desempeño del modelo y serán excluidas del entrenamiento.


- Algunas variables del dataset se encuentran almacenadas en tipos de datos que no representan correctamente su naturaleza estadística o temporal, por lo que es necesario realizar conversiones antes del análisis y modelado.


- Conversión de variables temporales desde `object` a formato `datetime`, especialmente:
  - `issue_d`
  - `earliest_cr_line`
  - `last_credit_pull_d`

- Conversión de variables categóricas ordinales y estructuradas:
  - `term`: actualmente almacenada como texto (`"36 months"`, `"60 months"`), debe convertirse a formato numérico entero.
  - `emp_length`: contiene categorías textuales (`"10+ years"`, `"2 years"`, etc.) y debe transformarse a una representación ordinal numérica.




In [ ]:
# ============================================
# Eliminación de variables irrelevantes,
# con data leakage y exceso de valores faltantes
# ============================================

cols_to_drop = [
    # ------------------------
    # Variables sin valor predictivo
    # ------------------------
    "id",
    "member_id",
    "url",

    # ------------------------
    # Variables con data leakage
    # ------------------------
    "recoveries",
    "collection_recovery_fee",
    "total_rec_prncp",
    "total_rec_int",
    "last_pymnt_amnt",
    "last_pymnt_d",
    "next_pymnt_d",
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 66)


In [ ]:
# -----------------------------------
# Convert date variables to datetime
# -----------------------------------

date_cols = [
    'issue_d',
    'earliest_cr_line',
    'last_credit_pull_d',
]

# Convert columns
for col in date_cols:
    
    df_browse[col] = pd.to_datetime(
        df_browse[col],
        format='%b-%Y',
        errors='coerce'
    )

# Check result
df_browse[date_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 3 columns):
 #   Column              Dtype         
---  ------              -----         
 0   issue_d             datetime64[ns]
 1   earliest_cr_line    datetime64[ns]
 2   last_credit_pull_d  datetime64[ns]
dtypes: datetime64[ns](3)
memory usage: 51.7 MB


## 1.4 Variables numéricas y catégoricas

In [ ]:
num_cols = df_browse.select_dtypes(include='number').columns.tolist()
cat_cols = df_browse.select_dtypes(include='object').columns.tolist()
date_cols = [c for c in df_browse.columns if 'd' in c]

In [ ]:
cat_summary = pd.DataFrame({
    'variable': cat_cols,
    'cardinalidad': [df_browse[col].nunique() for col in cat_cols],
})

cat_summary = cat_summary.sort_values(
    by='cardinalidad',
    ascending=False
)

cat_summary

,variable,cardinalidad
5,emp_title,512694
10,desc,124500
7,title,63154
12,zip_code,956
11,addr_state,51
2,sub_grade,35
9,purpose,14
8,emp_length,11
6,loan_status,9
15,grade,7


In [ ]:
import pandas as pd

# Seleccionar variables categóricas

summary = []

for col in cat_cols:
    
    vc = df_browse[col].value_counts(normalize=True, dropna=False)
    
    summary.append({
        'variable': col,
        'top_category': vc.index[0],
        'top_category_%': round(vc.iloc[0] * 100, 2),
        'second_category_%': round(vc.iloc[1] * 100, 2) if len(vc) > 1 else 0,
        'is_constant': vc.iloc[0] >= 0.999,
        'highly_imbalanced': vc.iloc[0] >= 0.95,
    })

cat_balance_summary = pd.DataFrame(summary)

cat_balance_summary = cat_balance_summary.sort_values(
    by='top_category_%',
    ascending=False
)

cat_balance_summary

,variable,top_category,top_category_%,second_category_%,is_constant,highly_imbalanced
14,pymnt_plan,n,99.97,0.03,True,True
16,verification_status_joint,NaN,94.88,2.54,False,False
1,application_type,Individual,94.66,5.34,False,False
10,desc,NaN,94.42,0.01,False,False
4,term,36 months,71.21,28.79,False,False
0,initial_list_status,w,67.92,32.08,False,False
9,purpose,debt_consolidation,56.53,22.87,False,False
7,title,Debt consolidation,51.01,20.78,False,False
3,home_ownership,MORTGAGE,49.16,39.59,False,False
6,loan_status,Fully Paid,47.63,38.85,False,False


In [ ]:
cols_to_drop = [
    # ------------------------
    # Variables con cardinalidad excesiva
    # ------------------------
    "emp_title",
    "title",
    "zip_code",

    # ------------------------
    # Variables constantes o excesivamente desbalanceadas
    # ------------------------

    "pymnt_plan"
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 57)


## 1.4 Análisis de datos faltantes y duplicados

In [ ]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
mths_since_last_delinq,1158535,51.246715
il_util,1068883,47.281042
mths_since_rcnt_il,909957,40.251099
all_util,866381,38.323555
open_acc_6m,866163,38.313912
total_cu_tl,866163,38.313912
inq_last_12m,866163,38.313912
max_bal_bc,866162,38.313868
open_il_12m,866162,38.313868
open_rv_12m,866162,38.313868


In [ ]:
df_browse.duplicated().sum()

np.int64(0)

- El dataset presenta 32 filas duplicadas que serán investigadas en el siguiente paso

- Se establecerá un umbral de eliminación del 70% de valores faltantes.
  - Las variables con un porcentaje igual o superior a este umbral serán eliminadas del análisis principal.
  - Esto se debe a que:
    - Su capacidad informativa es limitada.
    - La imputación podría introducir sesgos importantes.
    - Algunas corresponden a casos muy específicos o poco representativos.
    - Otras contienen información posterior a la aprobación del préstamo (*data leakage*).

  - Bajo este criterio, variables como:
    - `member_id`
    - `verification_status_joint`
    - `dti_joint`
    - `annual_inc_joint`
    - `desc`
    - `mths_since_last_record`
    - `mths_since_last_major_derog`
  
  serán excluidas del análisis principal.

- La presencia de valores faltantes puede deberse a distintas razones:
  - El solicitante no proporcionó la información.
  - La información no fue registrada correctamente.
  - El valor no aplicaba para ciertos clientes.
  - El valor faltante realmente representa ausencia de eventos financieros.
  - Cambios históricos en la captura de variables dentro de LendingClub.

- Dependiendo del comportamiento de los valores faltantes, se aplicarán distintas estrategias:
  - Eliminación de variables con:
    - porcentaje excesivo de nulos,
    - alta cardinalidad irrelevante,
    - o ausencia de valor predictivo.
  
  - Eliminación de observaciones:
    - En variables con porcentajes mínimos de valores faltantes (<1%), los registros afectados podrán eliminarse directamente debido al gran tamaño del dataset.
  
  - Imputación:
    - Se considerarán técnicas como:
      - media,
      - mediana,
      - moda,
      - o categorías como `"Unknown"` para variables categóricas.
  
  - Tratamiento del missing como información:
    - En ciertos casos, la ausencia de información puede estar asociada al perfil de riesgo del solicitante.
    - Por ejemplo, no tener historial crediticio podría representar un perfil financiero distinto.
    - En estos escenarios, el valor faltante podrá tratarse como una categoría adicional.

- Procederemos a hacer una investigación del comportamiendo de los y causa de los duplicados y valores faltantes

In [ ]:
cols_to_drop = [
    # ------------------------
    # Variables con más de 70% de missing
    # ------------------------
    "verification_status_joint",
    "dti_joint",
    "annual_inc_joint",
    "desc",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 57)


### 1.4.1 Investigando los NAs y duplicados

### 1.4.1.1 Duplicados

In [ ]:
duplicates = df_browse[df_browse.duplicated(keep=False)]

duplicates.sort_values(by="loan_status").head(32)

,int_rate,max_bal_bc,total_rec_late_fee,il_util,initial_list_status,tot_coll_amt,policy_code,application_type,total_pymnt_inv,sub_grade,fico_range_high,open_acc,out_prncp_inv,home_ownership,issue_d,term,open_il_12m,pub_rec,revol_bal,dti,last_fico_range_high,loan_status,installment,delinq_2yrs,last_fico_range_low,annual_inc,fico_range_low,emp_length,open_rv_12m,purpose,open_il_24m,addr_state,total_acc,open_acc_6m,funded_amnt,out_prncp,verification_status,loan_amnt,acc_now_delinq,revol_util,inq_fi,mths_since_last_delinq,inq_last_12m,pymnt_plan,open_rv_24m,total_pymnt,inq_last_6mths,total_cu_tl,collections_12_mths_ex_med,grade,all_util,funded_amnt_inv,total_bal_il,mths_since_rcnt_il,earliest_cr_line,last_credit_pull_d,tot_cur_bal


In [ ]:
# Remove fully empty and duplicated rows
df_browse = (
    df_browse
    .dropna(how='all')
    .drop_duplicates()
)

### 1.4.1.2 NAs

In [ ]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
mths_since_last_delinq,1158502,51.246003
il_util,1068850,47.280273
mths_since_rcnt_il,909924,40.250227
all_util,866348,38.322655
open_acc_6m,866130,38.313012
total_cu_tl,866130,38.313012
inq_last_12m,866130,38.313012
max_bal_bc,866129,38.312968
open_il_12m,866129,38.312968
open_rv_12m,866129,38.312968


In [ ]:
print("La distribución porcentual de la variable 'loan_status' es:\n")

print(
    (df_browse["loan_status"]
     .value_counts(normalize=True) * 100)
)

La distribución porcentual de la variable 'loan_status' es:

loan_status
Fully Paid                                             47.629771
Current                                                38.852100
Charged Off                                            11.879630
Late (31-120 days)                                      0.949587
In Grace Period                                         0.373164
Late (16-30 days)                                       0.192377
Does not meet the credit policy. Status:Fully Paid      0.087939
Does not meet the credit policy. Status:Charged Off     0.033663
Default                                                 0.001769
Name: proportion, dtype: float64


In [ ]:
print("La distribución porcentual de la variable 'loan_status' es:\n")

print(
    (df["loan_status"]
     .value_counts(normalize=True) * 100))

La distribución porcentual de la variable 'loan_status' es:

loan_status
Fully Paid                                             47.629771
Current                                                38.852100
Charged Off                                            11.879630
Late (31-120 days)                                      0.949587
In Grace Period                                         0.373164
Late (16-30 days)                                       0.192377
Does not meet the credit policy. Status:Fully Paid      0.087939
Does not meet the credit policy. Status:Charged Off     0.033663
Default                                                 0.001769
Name: proportion, dtype: float64


In [ ]:
df.zip_code.nunique()

956

In [ ]:
# Distribución global
global_dist = (
    df['loan_status']
    .value_counts(normalize=True)
    .rename("global")
)

# Distribución de missing
missing_dist = (
    df.loc[df['emp_length'].isna(), 'loan_status']
    .value_counts(normalize=True)
    .rename("missing")
)

pd.concat([global_dist, missing_dist], axis=1)

,global,missing
loan_status,,
Fully Paid,0.476298,0.390594
Current,0.388521,0.445656
Charged Off,0.118796,0.143832
Late (31-120 days),0.009496,0.012600
In Grace Period,0.003732,0.003996
Late (16-30 days),0.001924,0.003056
Does not meet the credit policy. Status:Fully Paid,0.000879,0.000129
Does not meet the credit policy. Status:Charged Off,0.000337,0.000102
Default,0.000018,0.000034
